# KORA 규정집 기반 Chroma DB 구축 및 Ragas 성능 평가 실습
본 노트북은 공부했던 `2.rag_with_chroma.ipynb` 실습 코드를 바탕으로 **KORA 규정집.pdf** 문서를 로드하여 Chroma DB를 구축하고, 구축된 RAG 시스템의 성능을 **Ragas** 프레임워크를 사용해 단일 평가해보는 전체 실습 공간입니다.

## Part 1. [실습 기반] 기본 RAG 시스템 구축

In [ ]:
# 1. 필요 패키지 설치
%pip install -qU langchain langchain-community langchain-core langchain-text-splitters langchain-openai langchain-chroma pypdf chromadb datasets ragas pandas matplotlib

In [ ]:
# 2. 기본 라이브러리 로드 및 환경변수 설정
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# LangChain & Chroma 관련 패키지
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# Ragas 및 Dataset 패키지
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy

# 환경변수 로드 (.env 파일이 루트 디렉토리에 있으므로, 상위 경로를 탐색하여 로드)
load_dotenv(dotenv_path="../../.env")

print("✅ 환경 변수 및 필수 라이브러리 로드 완료!")

In [ ]:
# 3. 문서 읽기 및 쪼개기
pdf_path = "../../docs/KORA 규정집.pdf"

if os.path.exists(pdf_path):
    loader = PyPDFLoader(pdf_path)
    document = loader.load()
    print(f"📖 문서 로드 성공! 총 {len(document)} 페이지")
    
    # RecursiveCharacterTextSplitter를 사용해 chunking 진행
    # 공부했던 실습 파라미터 적용: chunk_size=1500, chunk_overlap=200
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200
    )
    document_list = text_splitter.split_documents(document)
    print(f"✂️ 문서를 {len(document_list)}개의 청크(chunk)로 쪼갰습니다.")
else:
    print(f"❌ 파일을 찾을 수 없습니다. 경로를 확인해주세요: {pdf_path}")

In [ ]:
# 4. 임베딩 모델 정의
# OpenAI에서 제공하는 Embedding Model을 활용해서 chunk를 vector화
embedding = OpenAIEmbeddings(model='text-embedding-3-large')
print("✨ OpenAI Embeddings 모델 로드 완료!")

In [ ]:
# 5. 임베딩 벡터 데이터베이스(Chroma DB) 빌드 및 저장
# 로컬 디렉토리 './chroma_kora'에 collection_name='chroma-kora'로 저장
persist_dir = "./chroma_kora"

print("💾 Chroma DB 빌드 및 로컬 저장 중...")
database = Chroma.from_documents(
    documents=document_list,
    embedding=embedding,
    collection_name='chroma-kora',
    persist_directory=persist_dir
)
print(f"✅ 벡터 데이터베이스가 '{persist_dir}'에 정상적으로 저장되었습니다!")

In [ ]:
# 6. 이미 저장된 데이터를 불러와서 사용할 때
database = Chroma(
    collection_name='chroma-kora',
    persist_directory="./chroma_kora",
    embedding_function=embedding
)
print("🔍 로컬 Chroma DB 로드 완료!")

In [ ]:
# 7. 유사도 검색 테스트
query = "KORA의 복리후생 규정에 대해 알려주세요."
# k=3 개 추출
retrieved_docs = database.similarity_search(query, k=3)

print(f"❓ 질문: {query}")
print(f"🎯 검색된 관련 문서 수: {len(retrieved_docs)}")
print("\n--- [검색된 첫 번째 청크 내용 일부] ---")
if retrieved_docs:
    print(retrieved_docs[0].page_content[:500] + "...")

In [ ]:
# 8. 랭체인 LCEL을 활용한 qa_chain 구축 및 질의응답
llm = ChatOpenAI(model='gpt-4o', temperature=0)

custom_prompt = ChatPromptTemplate.from_template("""[Identity]
- 당신은 최고의 KORA 규정 전문가입니다.
- [Context]를 참고해서 사용자의 질문에 정확하고 친절하게 답변해주세요.
- 문서에 정보가 없는 경우 솔직하게 모른다고 대답하세요.

[Context]
{context}

Question: {question}
""")

# qa_chain 구성
qa_chain = (
    {
        "context": database.as_retriever(search_kwargs={'k': 3}),  # 질문과 유사한 문서를 DB에서 자동 검색해서 {context}에 넣음
        "question": RunnablePassthrough()                          # invoke()에 넣은 질문을 그대로 {question}에 넣음
    }
    | custom_prompt       # 위 두 값을 템플릿에 조합
    | llm                 # 완성된 프롬프트를 LLM에 전달
    | StrOutputParser()   # LLM 응답을 문자열로 변환
)

# 최종 체인 테스트
test_question = "임직원의 경조금 지급 기준은 어떻게 되나요?"
print(f"❓ 체인 테스트 질문: {test_question}")
response = qa_chain.invoke(test_question)
print("\n🤖 체인 응답:")
print(response)

---
## Part 2. Ragas를 이용한 구축된 RAG 성능 평가
방금 빌드한 `chunk_size=1500, chunk_overlap=200` 크기의 RAG 시스템에 대해서 **Ragas** 평가 데이터셋을 활용해 성능 지표를 검증하는 구간입니다.

In [ ]:
# 9. 평가 데이터셋 (kora_eval_dataset.json) 로드
print("--- 1. 평가용 데이터셋 로드 ---")
dataset_path = "../../docs/kora_eval_dataset.json"

try:
    with open(dataset_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    
    # JSON 구조에서 질문(question)과 모범답안(ground_truth) 추출
    test_questions = [item['question'] for item in eval_data['dataset']]
    ground_truths = [item['ground_truth'] for item in eval_data['dataset']]
    print(f"🎉 총 {len(test_questions)}개의 평가용 테스트 데이터셋이 정상 로드되었습니다!")
except FileNotFoundError:
    print(f"⚠️ {dataset_path} 파일을 찾을 수 없습니다. 평가 데이터셋을 먼저 준비해주세요.")
    test_questions = []
    ground_truths = []

In [ ]:
# 10. 구축한 qa_chain 실행 및 Ragas 평가용 데이터 (contexts, answer) 수집
print("--- 2. RAG 체인 답변 생성 및 컨텍스트 수집 중 (시간이 소요될 수 있습니다) ---")

if not test_questions:
    print("❌ 평가 데이터셋이 존재하지 않아 평가 데이터 준비를 건너뜁니다.")
else:
    # Ragas 채점용 DTO 세팅
    data_for_ragas = {
        "question": [],
        "contexts": [],
        "answer": [],
        "ground_truth": []
    }
    
    retriever = database.as_retriever(search_kwargs={'k': 3})
    
    # 각 질문별로 답변 생성 및 관련 컨텍스트 수집
    for i, q in enumerate(test_questions):
        print(f"[{i+1}/{len(test_questions)}] 질문 처리 중: {q}")
        
        # 1) 체인을 통한 답변 생성
        answer = qa_chain.invoke(q)
        
        # 2) 질문과 매칭되어 검색된 원본 문서들의 텍스트 리스트(contexts) 추출
        retrieved_docs = retriever.invoke(q)
        contexts = [doc.page_content for doc in retrieved_docs]
        
        # 3) Ragas DTO에 적재
        data_for_ragas["question"].append(q)
        data_for_ragas["contexts"].append(contexts)
        data_for_ragas["answer"].append(answer)
        data_for_ragas["ground_truth"].append(ground_truths[i])
        
    print("
✅ 평가용 RAG 생성 데이터 수집 완료!")

In [ ]:
# 11. Ragas evaluate 실행 및 점수 지표 계산
print("--- 3. Ragas 지표 채점 시작 ---")

if 'data_for_ragas' not in locals() or not test_questions:
    print("❌ 평가 데이터가 준비되지 않아 채점을 건너뜁니다.")
else:
    # Hugging Face Dataset 객체로 변환
    eval_dataset = Dataset.from_dict(data_for_ragas)
    
    # Ragas 핵심 4대 지표 평가 수행
    score = evaluate(
        dataset=eval_dataset,
        metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
    )
    
    print("
🎉 Ragas 성능 채점이 성공적으로 완료되었습니다!")

In [ ]:
# 12. 최종 평가 지표 출력 및 시각화
if 'score' in locals() and score:
    # 결과 점수 딕셔너리를 판다스 데이터프레임으로 보기 좋게 렌더링
    df_score = pd.DataFrame([score])
    df_score.index = ["Score (chunk_1500_200)"]
    
    print("📊 [Ragas 성능 평가 지표 표]")
    display(df_score.T)
    
    # 간단한 시각화 그래프 그리기
    plt.figure(figsize=(8, 5))
    plt.bar(df_score.columns, df_score.iloc[0], color=['dodgerblue', 'limegreen', 'orange', 'tomato'])
    plt.title('KORA RAG 시스템 성능 지표 (Chunk 1500 / Overlap 200)')
    plt.ylabel('Score (0.0 ~ 1.0)')
    plt.ylim(0, 1.1)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # 각 바 위에 점수 텍스트 표시
    for i, v in enumerate(df_score.iloc[0]):
        plt.text(i, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold')
        
    plt.show()
else:
    print("❌ 출력할 Ragas 평가 점수 데이터가 존재하지 않습니다.")